[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage4_sql.ipynb)

# 阶段四 · SQL 数据库（Day 16–19）

> 《Python 数据处理 20 天学习计划》阶段四配套教材。
> **开始前：文件 → 在云端硬盘中保存一份副本。**

**本阶段目标**：会用 SQL 查询数据库，并能在 SQL 和 Pandas 之间自由搬运数据。

**为什么用 SQLite**：它是「一个文件就是一个数据库」的轻量数据库，Python 自带、Colab 里直接运行，**不需要安装任何数据库服务器**——这正是本计划对「服务器」的全部要求。

## Day 16 · 认识数据库 + 第一个 SELECT

核心概念一句话：**数据库里有若干张表，每张表由行和列组成**。SQL 就是操作这些表的语言。

下面用 Python 自带的 `sqlite3` 建库、建表、插入数据、查询。

In [ ]:
import sqlite3

# 连接数据库：文件不存在会自动创建（一个文件就是一个数据库）
conn = sqlite3.connect("school.db")
cur = conn.cursor()

# 建表（先删后建，方便重复运行本单元格）
cur.execute("DROP TABLE IF EXISTS students")
create_sql = """
CREATE TABLE students (
    id      INTEGER PRIMARY KEY,
    name    TEXT,
    class   TEXT,
    math    INTEGER,
    english INTEGER
)
"""
cur.execute(create_sql)

# 插入数据
rows = [
    (101, "小明", "一班", 88, 92),
    (102, "小红", "二班", 95, 85),
    (103, "小刚", "一班", 62, 70),
    (104, "小丽", "二班", 77, 88),
    (105, "小华", "一班", 85, 90),
]
cur.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?)", rows)
conn.commit()   # 提交保存
print("数据写入成功")

In [ ]:
# 第一个查询：SELECT 列 FROM 表
# cur.execute 执行 SQL，fetchall 取回所有结果
cur.execute("SELECT * FROM students")   # * 表示所有列
for row in cur.fetchall():
    print(row)

In [ ]:
# 只查部分列 + LIMIT 限制行数
cur.execute("SELECT name, math FROM students LIMIT 3")
cur.fetchall()

### ✏️ 练习 16-1
查询 students 表中所有同学的 **姓名和班级** 两列（提示：`SELECT name, class FROM students`）。

### ✏️ 练习 16-2
向表中再插入一位新同学：`(106, "小芳", "二班", 90, 86)`，插入后 `SELECT * FROM students` 验证（提示：用 `cur.execute("INSERT INTO students (id, name, class, math, english) VALUES (?,?,?,?,?)", (106, "小芳", "二班", 90, 86))` 然后 `conn.commit()`）。

In [ ]:
# 练习 16-1：在这里写你的代码

In [ ]:
# 练习 16-2：在这里写你的代码

## Day 17 · WHERE 过滤 + GROUP BY 聚合

| 语法 | 作用 | 例子 |
|---|---|---|
| `WHERE` | 行级过滤 | `WHERE math > 80` |
| `AND / OR` | 组合条件 | `WHERE math > 80 AND class = '一班'` |
| `LIKE` | 文本模糊匹配 | `WHERE name LIKE '小%'`（% 是通配符） |
| `ORDER BY` | 排序 | `ORDER BY math DESC`（DESC 降序） |
| `GROUP BY` | 分组聚合 | `GROUP BY class` 配合 `COUNT/AVG/SUM` |

> 对照阶段二：WHERE ≈ 布尔筛选，GROUP BY ≈ groupby，ORDER BY ≈ sort_values。

In [ ]:
# WHERE：数学大于 80 的同学
cur.execute("SELECT name, math FROM students WHERE math > 80")
cur.fetchall()

In [ ]:
# 多条件 + 模糊匹配：一班且姓“小”的同学（LIKE '小%' 表示以“小”开头）
cur.execute("SELECT name, class, math FROM students WHERE class = '一班' AND name LIKE '小%'")
cur.fetchall()

In [ ]:
# GROUP BY：每个班的人数和平均数学成绩（ROUND 保留 1 位小数）
group_sql = """
SELECT class, COUNT(*) AS 人数, ROUND(AVG(math), 1) AS 平均数学
FROM students
GROUP BY class
ORDER BY 平均数学 DESC
"""
cur.execute(group_sql)
cur.fetchall()

### ✏️ 练习 17-1
查询英语成绩在 85 分以上（含 85）的同学，按英语成绩从高到低排序，显示姓名和英语。

### ✏️ 练习 17-2
统计每个班的**英语最高分**（提示：`MAX(english)`，配合 GROUP BY）。

In [ ]:
# 练习 17-1：在这里写你的代码

In [ ]:
# 练习 17-2：在这里写你的代码

## Day 18 · 多表 JOIN + NULL 处理

- **JOIN**：把两张表按共同字段（键）连接起来——和 Day 9 的 `pd.merge` 是同一思想。
- **NULL**：数据库里的「空/未知」。判断空值**必须用 `IS NULL`**，写 `= NULL` 是查不到的（经典易错点）。

In [ ]:
# 再建一张语文成绩表（学生和成绩分开存放，是数据库的常见设计）
cur.execute("DROP TABLE IF EXISTS chinese_scores")
cur.execute("CREATE TABLE chinese_scores (student_id INTEGER, chinese INTEGER)")
cur.executemany("INSERT INTO chinese_scores VALUES (?, ?)",
                [(101, 85), (102, 90), (103, 66), (104, 82), (105, 88), (106, 91)])
conn.commit()

# INNER JOIN：按 id 把两张表连起来（s 和 c 是表的别名，写起来短）
join_sql = """
SELECT s.name, s.class, c.chinese
FROM students s
JOIN chinese_scores c ON s.id = c.student_id
"""
cur.execute(join_sql)
cur.fetchall()

In [ ]:
# LEFT JOIN：左表的学生全部保留，右表没匹配上的显示 NULL
cur.execute("""
SELECT s.name, c.chinese
FROM students s
LEFT JOIN chinese_scores c ON s.id = c.student_id
""")
cur.fetchall()

In [ ]:
# 制造一个 NULL 并学习如何判断
cur.execute("ALTER TABLE students ADD COLUMN phone TEXT")   # 给表加一列，默认全是 NULL
conn.commit()

# 找出没有填写电话的同学：必须用 IS NULL
cur.execute("SELECT name FROM students WHERE phone IS NULL")
cur.fetchall()

### ✏️ 练习 18-1
用 JOIN 查询每个同学的姓名、班级、数学和语文成绩，只保留语文成绩大于 80 的同学（提示：JOIN 之后加 `WHERE c.chinese > 80`）。

### ✏️ 练习 18-2
查询**已填写**电话的同学（提示：`IS NOT NULL`）。

In [ ]:
# 练习 18-1：在这里写你的代码

In [ ]:
# 练习 18-2：在这里写你的代码

## Day 19 · SQL ↔ Pandas 互通

到这一步，两个世界就打通了：

- `pd.read_sql_query("SQL语句", conn)`：查询结果直接变成 **DataFrame**；
- `df.to_sql("表名", conn)`：把 DataFrame 写回数据库变成**一张表**。

这是实际工作中最常见的组合：**SQL 负责取数，Pandas 负责精细加工**。

In [ ]:
import pandas as pd

# SQL 取数 → DataFrame：一行代码
df_sql = pd.read_sql_query("""
SELECT s.name, s.class, s.math, s.english, c.chinese
FROM students s
JOIN chinese_scores c ON s.id = c.student_id
""", conn)
df_sql

In [ ]:
# Pandas 加工：算总分（这一步在 Pandas 里写起来更顺手）
df_sql["总分"] = df_sql["math"] + df_sql["english"] + df_sql["chinese"]

# 写回数据库，变成一张新表
df_sql.to_sql("total_scores", conn, if_exists="replace", index=False)

# 验证：直接从数据库查这张新表
pd.read_sql_query("SELECT * FROM total_scores ORDER BY 总分 DESC", conn)

In [ ]:
# 选学：窗口函数——不折叠行数的“排名”
# RANK() OVER (ORDER BY math DESC)：按数学成绩给每人排名
pd.read_sql_query("""
SELECT name, math,
       RANK() OVER (ORDER BY math DESC) AS 数学排名
FROM students
""", conn)

### ✏️ 练习 19-1
用 `read_sql_query` 查询每个班的平均分（数学、英语、语文三科总平均，用 `AVG(s.math + s.english + c.chinese)`），结果按平均分从高到低排序，返回 DataFrame。

In [ ]:
# 练习 19-1：在这里写你的代码

In [ ]:
# 全部学完后，可以关闭数据库连接（好习惯）
# 注意：关闭后上面的查询就不能再执行了，需要时重新运行 Day 16 第一个单元格
# conn.close()
print("阶段四学习完成，记得做测验 4！")

## Day 19 · 缓冲日：补漏 + 互动答疑 + 测验 4

**今天不新学内容，安排如下：**

1. **补漏（约 1 小时）**：把 Day 16–19 没完成的部分补齐；
2. **整理问题清单（约 15 分钟）**：翻看本 notebook 末尾的「我的问题清单」，已解决的标记掉，没解决的按主题归类；
3. **互动答疑（约 30 分钟）**：和老师 / 同学 / 辅导者约一次答疑，照着问题清单逐条问，当场把答案记进清单；
4. **完成测验 4**：打开网页版学习计划中的 <b>quiz4.html</b>（10 题，即时判分含解析，60 合格 / 80 优秀）。

> 💡 **平时怎么记录问题？** 每天学习中一遇到卡住的地方，立刻在问题清单里记一行：日期、做了什么、报错信息或困惑点。问题越具体，答疑效率越高——「groupby 之后取某一列报错 KeyError」比「groupby 不会」好问得多。

---

## 📒 我的问题清单（随手记录，缓冲日答疑前整理）

> 使用方法：学习中遇到任何卡住的地方，双击本单元格，在表格里加一行。答疑后把「状态」改为已解决并写下答案要点。

| 日期 | 遇到的问题 / 报错信息 | 状态（待问 / 已解决） | 答案要点 |
|---|---|---|---|
| 示例 | groupby 之后想取某一列却报错 KeyError | 已解决 | 结果是 Series，要用 reset_index() 还原成表 |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---

## ✅ 参考答案（先独立完成，再对照）

In [ ]:
# 练习 16-1
cur.execute("SELECT name, class FROM students")
cur.fetchall()

In [ ]:
# 练习 16-2（如果之前已经插入过 106 号，重复运行会报错，属正常现象）
try:
    cur.execute("INSERT INTO students (id, name, class, math, english) VALUES (?,?,?,?,?)", (106, "小芳", "二班", 90, 86))
    conn.commit()
except Exception as e:
    print("提示：", e)
cur.execute("SELECT * FROM students")
cur.fetchall()

In [ ]:
# 练习 17-1
cur.execute("SELECT name, english FROM students WHERE english >= 85 ORDER BY english DESC")
print(cur.fetchall())

# 练习 17-2
cur.execute("SELECT class, MAX(english) FROM students GROUP BY class")
print(cur.fetchall())

In [ ]:
# 练习 18-1
cur.execute("""
SELECT s.name, s.class, s.math, c.chinese
FROM students s
JOIN chinese_scores c ON s.id = c.student_id
WHERE c.chinese > 80
""")
print(cur.fetchall())

# 练习 18-2
cur.execute("SELECT name FROM students WHERE phone IS NOT NULL")
print(cur.fetchall())   # 目前没人填电话，结果为空列表是正常的

In [ ]:
# 练习 19-1
pd.read_sql_query("""
SELECT s.class,
       ROUND(AVG(s.math + s.english + c.chinese), 1) AS 总平均
FROM students s
JOIN chinese_scores c ON s.id = c.student_id
GROUP BY s.class
ORDER BY 总平均 DESC
""", conn)

---

## 🎉 阶段四完成，下一站

1. 回到网页版学习计划，完成 **测验 4（quiz4.html）**；
2. 打开 [阶段五 · 综合项目](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage5_project.ipynb)，用 Day 20 把 20 天所学串起来。